Note: to run this noteboook, download the CCDC Python CSD and create the dedicated conda environment as follows:
```bash
conda create -n ccdc-env
conda activate ccdc-env
conda install --channel=https://conda.ccdc.cam.ac.uk csd-python-api
conda install ipykernel
```

In [ ]:
#this is reading all the molecule IDs in our file
#note: your file has to be in the exact right directory and the name has to be correct
#additional functionality to add if there's time: add your own file from any path

import pandas as pd

data = pd.read_csv("molecules.csv")

ids = data['Database identifier'].tolist()

print(ids)


['HXACAN06', 'ACSALA02', 'IBPRAC03', 'BAQFOS', 'DOVGEC02', 'WARFAR10', 'MELATN01', 'COCAIN10', 'ABULIT', 'ALESOC', 'FAFYIZ', 'GEKKIU', 'ACSALA10']


In [52]:
#this is making a list of all the molecule IDs in our file
for identifier in ids:
    print(identifier)

HXACAN06
ACSALA02
IBPRAC03
BAQFOS
DOVGEC02
WARFAR10
MELATN01
COCAIN10
ABULIT
ALESOC
FAFYIZ
GEKKIU
ACSALA10


In [ ]:
#this makes SDF files of all the molecules in our csv file based on their IDs, while also keeping track of which ones were successfully fetched

from pathlib import Path
import pandas as pd
from ccdc import io
from ccdc.io import MoleculeWriter

input_csv = "molecules.csv"
output_csv = "molecules_with_status.csv"

df = pd.read_csv(input_csv)

out_dir = Path("sdf_files")
out_dir.mkdir(exist_ok=True)    

reader = io.EntryReader("CSD")

status_list = []
message_list = []
sdf_path_list = []

for identifier in df['Database identifier']:
    identifier = str(identifier).strip()
    out_path = out_dir / f"{identifier}.sdf"
    sdf_path_list.append(str(out_path))

    try:
       entry = reader.entry(identifier)

       mol = entry.molecule
       if len(mol.components) > 1:
            mol = max(mol.components, key=lambda c: len(list(c.atoms)))

       with io.MoleculeWriter(str(out_path)) as mol_writer:
           mol_writer.write(mol)

       if out_path.exists():
            status_list.append("processed")
            message_list.append("")
            print(f"Wrote {identifier} to {out_path}")
       else:
            status_list.append("failed")
            message_list.append("SDF file was not created")
            print(f"Failed to write {identifier}")

    except Exception as err:
        status_list.append("failed")
        message_list.append(str(err))
        print(f"Could not process {identifier}: {err}")

df["SDF file"] = sdf_path_list
df["Processing status"] = status_list
df["Processing message"] = message_list

number_processed = status_list.count("processed")
number_failed = status_list.count("failed")  

df["Number processed"] = number_processed
df["Number failed"] = number_failed
           
df.to_csv(output_csv, index=False)

print(f"Saved updated CSV to {output_csv}")
print(f"Number of molecules processed: {number_processed}")
print(f"Number of molecules failed: {number_failed}")



Wrote HXACAN06 to sdf_files\HXACAN06.sdf
Wrote ACSALA02 to sdf_files\ACSALA02.sdf
Wrote IBPRAC03 to sdf_files\IBPRAC03.sdf
Wrote BAQFOS to sdf_files\BAQFOS.sdf
Wrote DOVGEC02 to sdf_files\DOVGEC02.sdf
Wrote WARFAR10 to sdf_files\WARFAR10.sdf
Wrote MELATN01 to sdf_files\MELATN01.sdf
Wrote COCAIN10 to sdf_files\COCAIN10.sdf
Wrote ABULIT to sdf_files\ABULIT.sdf
Wrote ALESOC to sdf_files\ALESOC.sdf
Could not process FAFYIZ: DatabasePool::entry( DatabaseEntryIdentifier )(): FAFYIZ is not in the database
Wrote GEKKIU to sdf_files\GEKKIU.sdf
Wrote ACSALA10 to sdf_files\ACSALA10.sdf
Saved updated CSV to molecules_with_status.csv
Number of molecules processed: 12
Number of molecules failed: 1


In [54]:
from pathlib import Path

print(Path.cwd())

c:\Users\teaching\Documents\ccdc\ccdc
